# ClaimWise Insurance Prediction — 15. DBSCAN (Density-Based Clustering)

## 1. Objective

Run **DBSCAN** on the actual ClaimWise features and report what density-based clustering finds:
number of clusters, noise points, cluster distribution and silhouette score.

| Rule | How it is respected |
|---|---|
| `loss` is not an input | dropped before fitting (printed below) |
| `eps` is not hand-picked | selected from the k-distance curve with a reproducible knee rule |
| sample is justified | DBSCAN's neighbourhood queries cost grows quickly with rows; a fixed, seeded 5,000-row sample is used and stated |
| PCA | used only for the 2D picture; DBSCAN itself runs on the scaled features |

Artifacts go to `Outputs/dbscan/`.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from academic_common import ensure_dirs, save_figure, style_axes

from academic_common import DBSCAN_DIR

ensure_dirs()
print("DBSCAN output folder:", DBSCAN_DIR)

DBSCAN output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Load the ClaimWise Features (target excluded)

In [4]:
# The real ClaimWise preprocessed data. The target "loss" is REMOVED: unsupervised methods
# must not see the target as an input feature.
df = pd.read_csv(DATASET_DIR / "05_preprocessed_final.csv")
target_present = "loss" in df.columns
features = df.drop(columns=["loss"]) if target_present else df
loss_only = df["loss"] if target_present else None

print("raw preprocessed shape:", df.shape)
print("features used:", features.shape, "| 'loss' excluded from the inputs:", target_present)
print("features numeric:", all(pd.api.types.is_numeric_dtype(d) for d in features.dtypes))
print("missing values:", int(features.isna().sum().sum()))
print("duplicate rows:", int(features.duplicated().sum()))

raw preprocessed shape: (50000, 131)
features used: (50000, 130) | 'loss' excluded from the inputs: True
features numeric: True
missing values: 0


duplicate rows: 0


## 4. Sample and Scaling

A fixed sample of 5,000 rows (`random_state=42`) is drawn: DBSCAN compares every point with its
neighbours, and running it on all 50,000 × 130 values is unnecessary for the analysis and much more
expensive. The sample size is stated in every result below so nothing is hidden.

`StandardScaler` is fitted on the sample (unsupervised, no target involved).

In [5]:
from sklearn.preprocessing import StandardScaler

sample_idx = np.random.RandomState(42).choice(len(features), size=5000, replace=False)
X_sample_raw = features.iloc[sample_idx]
scaler = StandardScaler()
X_sample = scaler.fit_transform(X_sample_raw)
print("sample:", X_sample.shape, "| from rows:", sample_idx.min(), "...", sample_idx.max())

sample: (5000, 130) | from rows: 4 ... 49988


## 5. Choosing `eps` from the k-Distance Curve

DBSCAN needs `eps` (neighbourhood radius) and `min_samples` (minimum points to form a dense region).
`min_samples = 10` is fixed here. `eps` is taken from the **k-distance curve**: sort each point's
distance to its 10th nearest neighbour and use the *knee* of that curve — the standard, reproducible
rule (the knee below is found as the point with the largest perpendicular distance to the line
joining the curve's endpoints, so it is computed, not eyeballed).

In [6]:
from sklearn.neighbors import NearestNeighbors

MIN_SAMPLES = 10


def kdistances(X, k):
    distances, _ = NearestNeighbors(n_neighbors=k).fit(X).kneighbors(X)
    return np.sort(distances[:, -1])


def knee_eps(sorted_distances):
    y = (sorted_distances - sorted_distances.min()) / (sorted_distances.max() - sorted_distances.min())
    x = np.linspace(0, 1, len(y))
    line = y[0] + (y[-1] - y[0]) * x
    index = int(np.argmax(line - y))
    return float(sorted_distances[index]), index


kd = kdistances(X_sample, MIN_SAMPLES)
eps_130, knee_index = knee_eps(kd)
print(f"k-distance knee at index {knee_index}/{len(kd)} -> eps = {eps_130:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(kd, color="#D4512A", linewidth=1.8)
ax.axvline(knee_index, linestyle="--", color="#18181B",
           label=f"knee (eps = {eps_130:.3f})")
style_axes(ax, "k-distance curve — 10th nearest neighbour (130 scaled features)",
           "points sorted by distance", "distance to 10th neighbour")
ax.legend()
path = save_figure(fig, DBSCAN_DIR / "dbscan_k_distance_130d.png")
print("Saved:", path)

k-distance knee at index 4737/5000 -> eps = 13.6748
Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan/dbscan_k_distance_130d.png


In [7]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score


def run_dbscan(X, eps, min_samples=MIN_SAMPLES):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    unique = sorted(set(labels))
    n_clusters = len([c for c in unique if c != -1])
    noise_mask = labels == -1
    result = {
        "clusters": int(n_clusters),
        "noise_points": int(noise_mask.sum()),
        "noise_%": round(float(noise_mask.mean() * 100), 2),
        "eps": round(float(eps), 4),
        "min_samples": min_samples,
        "silhouette": None,
    }
    if n_clusters > 1 and (~noise_mask).sum() > n_clusters:
        result["silhouette"] = round(
            float(silhouette_score(X, labels, sample_size=3000, random_state=42)), 4
        )
    return labels, result


labels_130, result_130 = run_dbscan(X_sample, eps_130)
print("DBSCAN on the 130 scaled features:", result_130)
sizes_130 = pd.Series(labels_130[labels_130 >= 0]).value_counts().sort_index()
print("cluster sizes:", sizes_130.to_dict())

DBSCAN on the 130 scaled features: {'clusters': 7, 'noise_points': 164, 'noise_%': 3.28, 'eps': 13.6748, 'min_samples': 10, 'silhouette': 0.2617}
cluster sizes: {0: 4734, 1: 21, 2: 15, 3: 16, 4: 25, 5: 11, 6: 14}


## 6. Second View — DBSCAN After PCA (20 components)

Distances lose meaning in high dimensions (distance concentration), which often degenerates
density-based clustering. The same procedure is therefore repeated on the first **20 principal
components**, which keep most of the feature variance while removing noisy dimensions.

This is a genuine second experiment (its own scaler/PCA fitted on the sample, its own knee-selected
`eps`), reported side by side with the 130-feature run. The 2D plot further down is visualisation
only.

In [8]:
from sklearn.decomposition import PCA

pca20 = PCA(n_components=20, random_state=42)
X_pca20 = pca20.fit_transform(X_sample)
print("20 components explain {:.2%} of the sample variance".format(
    pca20.explained_variance_ratio_.sum()))

kd_pca = kdistances(X_pca20, MIN_SAMPLES)
eps_pca, knee_pca = knee_eps(kd_pca)
print(f"PCA-20 k-distance knee at index {knee_pca}/{len(kd_pca)} -> eps = {eps_pca:.4f}")

labels_pca, result_pca = run_dbscan(X_pca20, eps_pca)
print("DBSCAN on PCA-20:", result_pca)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(kd_pca, color="#E58A3A", linewidth=1.8)
ax.axvline(knee_pca, linestyle="--", color="#18181B", label=f"knee (eps = {eps_pca:.3f})")
style_axes(ax, "k-distance curve — 10th nearest neighbour (PCA-20)",
           "points sorted by distance", "distance to 10th neighbour")
ax.legend()
path = save_figure(fig, DBSCAN_DIR / "dbscan_k_distance_pca20.png")
print("Saved:", path)

20 components explain 51.05% of the sample variance
PCA-20 k-distance knee at index 4841/5000 -> eps = 8.8960
DBSCAN on PCA-20: {'clusters': 3, 'noise_points': 67, 'noise_%': 1.34, 'eps': 8.896, 'min_samples': 10, 'silhouette': 0.3675}


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan/dbscan_k_distance_pca20.png


In [9]:
summary = pd.DataFrame([result_130, result_pca])
summary.insert(0, "representation", ["130 scaled features", "PCA (20 components)"])
summary_path = DBSCAN_DIR / "dbscan_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan/dbscan_summary.csv


,representation,clusters,noise_points,noise_%,eps,min_samples,silhouette
0,130 scaled features,7,164,3.28,13.6748,10,0.2617
1,PCA (20 components),3,67,1.34,8.8960,10,0.3675


## 7. Visualisation of the DBSCAN Result (PCA for plotting only)

Noise points (label −1) are drawn in light grey; the coloured groups are the clusters found by
DBSCAN in the **original 130-dimensional** scaled sample.

In [10]:
projection = PCA(n_components=2, random_state=42).fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(9, 6.4))
noise = labels_130 == -1
ax.scatter(projection[noise, 0], projection[noise, 1], c="#D6CEC2", s=12,
           label=f"noise ({noise.sum()} points)", alpha=0.8)
clustered = ~noise
ax.scatter(projection[clustered, 0], projection[clustered, 1],
           c=labels_130[clustered], cmap="tab10", s=14, alpha=0.85, label="cluster")
style_axes(ax, "DBSCAN result in a 2D PCA projection (PCA = visualisation only)",
           "PC1", "PC2")
ax.legend(loc="best", fontsize=9)
path = save_figure(fig, DBSCAN_DIR / "dbscan_clusters_pca_visualization.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan/dbscan_clusters_pca_visualization.png


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

counts = pd.Series(labels_130[labels_130 >= 0]).value_counts().sort_index()
bars = axes[0].bar([f"cluster {c}" for c in counts.index], counts.values, color="#D4512A")
style_axes(axes[0], "DBSCAN cluster distribution (130 features)", "", "records")
axes[0].tick_params(axis="x", rotation=45)
for b in bars:
    b.set_edgecolor("#FFFFFF")
axes[1].bar(["noise", "clustered"],
            [int(noise.sum()), int((~noise).sum())],
            color=["#D6CEC2", "#3F7D4A"])
style_axes(axes[1], "Noise vs clustered points", "", "records")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%d", fontsize=10, padding=2)

fig.tight_layout()
path = save_figure(fig, DBSCAN_DIR / "dbscan_cluster_distribution.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/dbscan/dbscan_cluster_distribution.png


## 8. Conclusion

Actual results of this executed run (5,000-row sample, `min_samples = 10`, knee-selected `eps`):

| Representation | Clusters | Noise points | Noise % | Silhouette |
|---|---:|---:|---:|---:|
| 130 scaled features | 7 | 164 | 3.28 | 0.2617 |
| PCA (20 components) | 3 | 67 | 1.34 | 0.3675 |

Reading: DBSCAN finds density-based groups **and** marks records that fit no group as *noise*.
Noise here means "unusual in the feature space", not fraud and not an error — ClaimWise contains no
fraud label to support any such claim.

Artifacts: `dbscan_k_distance_130d.png`, `dbscan_k_distance_pca20.png`,
`dbscan_clusters_pca_visualization.png`, `dbscan_cluster_distribution.png`, `dbscan_summary.csv`.